In [1]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd

# ---------- User-configurable paths ----------
path_data_intermediate = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change me
os.makedirs(path_data_intermediate, exist_ok=True)

csv_out     = os.path.join(path_data_intermediate, "CCMLinkingTable.csv")
parquet_out = os.path.join(path_data_intermediate, "CCMLinkingTable.parquet")


In [2]:
#1 Load Compustat Data

SQL = """
SELECT
    a.gvkey,
    a.conm,
    a.tic,
    a.cusip,
    a.cik,
    a.sic,
    a.naics,
    b.linkprim,
    b.linktype,
    b.liid,
    b.lpermno,
    b.lpermco,
    b.linkdt,
    b.linkenddt
FROM comp.names AS a
INNER JOIN crsp.ccmxpf_lnkhist AS b
    ON a.gvkey = b.gvkey
WHERE b.linktype IN ('LC','LU')
  AND b.linkprim IN ('P','C')
  AND b.linkdt >= DATE '2000-01-01'
ORDER BY a.gvkey;
"""


In [3]:
#2 Compustat Data Extraction From WRDS
db = wrds.Connection()  
df = db.raw_sql(SQL, date_cols=["linkdt", "linkenddt"])

Enter your WRDS username [nglei]: nglei2025
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [5]:
#3 Clean Data

df = df.rename(
    columns={
        "linkdt": "timeLinkStart_d",
        "linkenddt": "timeLinkEnd_d",
        "lpermno": "permno",
    }
)

df = df.sort_values(["gvkey", "timeLinkStart_d", "timeLinkEnd_d"], kind="mergesort")

df.to_csv(csv_out, index=False)
df.to_parquet(parquet_out, index=False)

print("Saved:")
print(" -", csv_out)
print(" -", parquet_out)
print(df.head())

Saved:
 - /Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate/CCMLinkingTable.csv
 - /Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate/CCMLinkingTable.parquet
    gvkey                         conm   tic      cusip         cik   sic  \
0  001045  AMERICAN AIRLINES GROUP INC   AAL  02376R102  0000006201  4512   
1  001076            PROG HOLDINGS INC   PRG  74319R101  0001808834  6141   
2  001117         BK TECHNOLOGIES CORP  BKTI  05587G203  0000002186  3663   
3  001164                      MCI INC  MCIP  552691107  0000723527  4813   
4  001177                    AETNA INC   AET  00817Y108  0001122304  6324   

    naics linkprim linktype liid   permno  lpermco timeLinkStart_d  \
0  481111        P       LC   04  21020.0  20010.0      2013-12-09   
1  522220        P       LC   01  10517.0   5674.0      2010-12-01   
2  334220        P       LC   02  10779.0     65.0      2005